# Goldset Final Grade 결정

3번의 LLM Judge 결과를 합산하여 최종 grade를 확정하고,
query_id별 grade 2~3 문서 충분성을 점검한다.

## 판정 규칙
| 조건 | 결과 | confidence |
|---|---|---|
| 완전 일치 (3/3 동일) | gold 포함 | HIGH |
| 다수결 (2/3 동일, outlier 차이 무관) | gold 포함 | MEDIUM |
| 완전 불일치 (3개 모두 다른 값) | gold 제외 | LOW |

## 데이터 소스
| Judge | 파일 |
|---|---|
| 1st | `goldset_all_scenarios1.json` |
| 2nd | `goldset_all_scenarios2.json` |
| 3rd | `goldset_all_scenarios3.json` |

## 1. 데이터 로드

In [3]:
import json
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

DATA_DIR = Path("../dataset")

def load_json(path: Path) -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

judge1 = load_json(DATA_DIR / "goldset_all_scenarios1.json")
judge2  = load_json(DATA_DIR / "goldset_all_scenarios2.json")
judge3 = load_json(DATA_DIR / "goldset_all_scenarios3.json")

print(f"1st judge: {len(judge1):,}건")
print(f"2nd judge: {len(judge2):,}건" )
print(f"3rd judge: {len(judge3):,}건")

1st judge: 3,360건
2nd judge: 3,360건
3rd judge: 3,360건


In [4]:
# 각 Judge 파일의 Failed(relevance_grade=None) 건수 확인
from collections import defaultdict

def failed_summary(records: list, label: str):
    total  = len(records)
    failed = [r for r in records if r.get("relevance_grade") is None]
    by_qid = defaultdict(int)
    for r in failed:
        by_qid[r["query_id"]] += 1

    print(f"=== {label}  (총 {total:,}건 / 실패 {len(failed):,}건) ===")
    if failed:
        print(f"  {'query_id':>10}  {'failed':>7}")
        print(f"  {'-'*20}")
        for qid in sorted(by_qid):
            print(f"  {qid:>10}  {by_qid[qid]:>7}")
    else:
        print("  실패 없음")
    print()

failed_summary(judge1, "1st judge  (goldset_all_scenarios1.json)")
failed_summary(judge2, "2nd judge  (goldset_all_scenarios2.json)")
failed_summary(judge3, "3rd judge  (goldset_all_scenarios3.json)")

=== 1st judge  (goldset_all_scenarios1.json)  (총 3,360건 / 실패 0건) ===
  실패 없음

=== 2nd judge  (goldset_all_scenarios2.json)  (총 3,360건 / 실패 0건) ===
  실패 없음

=== 3rd judge  (goldset_all_scenarios3.json)  (총 3,360건 / 실패 0건) ===
  실패 없음



## 2. 3개 Judge 결과 병합

In [5]:
# isbn + query_id → grade 매핑 (None 이외는 모두 int로 정규화)
def to_int_grade(v):
    return int(v) if v is not None else None

g1 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge1}
g2 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge2}
g3 = {(r["isbn"], r["query_id"]): to_int_grade(r["relevance_grade"]) for r in judge3}

all_keys = set(g1) | set(g2) | set(g3)
missing  = {k for k in all_keys if not (k in g1 and k in g2 and k in g3)}
print(f"전체 문서 수: {len(all_keys):,}")
print(f"3개 judge 모두 있는 문서: {len(all_keys) - len(missing):,}")
if missing:
    print(f"⚠️  일부 judge 누락: {len(missing)}건 — 해당 문서는 제외됩니다")

전체 문서 수: 3,360
3개 judge 모두 있는 문서: 3,360


## 3. 최종 Grade 결정 함수

In [6]:
def decide_final_grade(grades: List[Optional[int]]) -> Dict[str, Any]:
    # None(LLM 오류) 포함 시 판정 불가 → 제외
    if any(g is None for g in grades):
        return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

    counter = Counter(grades)
    majority_grade, majority_count = counter.most_common(1)[0]

    # 완전 일치 (3/3) → HIGH
    if majority_count == 3:
        return {"final_grade": majority_grade, "confidence": "HIGH",   "gold_status": "included"}

    # 다수결 (2/3 동일 값) → MEDIUM  ※ outlier와의 차이는 무관
    if majority_count == 2:
        return {"final_grade": majority_grade, "confidence": "MEDIUM", "gold_status": "included"}

    # 완전 불일치 (3개 모두 다른 값) → 제외
    return {"final_grade": None, "confidence": "LOW", "gold_status": "excluded"}

## 4. 전체 문서에 판정 적용

In [7]:
# 메타데이터는 1st judge 기준으로 사용
meta = {(r["isbn"], r["query_id"]): r for r in judge1}

records = []
for key in sorted(all_keys - missing):
    isbn, query_id = key
    grades = [g1[key], g2[key], g3[key]]
    verdict = decide_final_grade(grades)

    base = meta[key]
    records.append({
        "query_id"   : query_id,
        "isbn"       : isbn,
        "title"      : base.get("title", ""),
        "grade_j1"   : grades[0],
        "grade_j2"   : grades[1],
        "grade_j3"   : grades[2],
        **verdict,
    })

df = pd.DataFrame(records)
print(df["gold_status"].value_counts().to_string())
print(f"\n포함: {(df['gold_status']=='included').sum():,}  /  제외: {(df['gold_status']=='excluded').sum():,}")

gold_status
included    3322
excluded      38

포함: 3,322  /  제외: 38


## 5. 판정 결과 분포

In [8]:
print("=== confidence 분포 ===")
print(df["confidence"].value_counts().to_string())

print("\n=== final_grade 분포 (포함된 문서) ===")
included = df[df["gold_status"] == "included"]
print(included["final_grade"].value_counts().sort_index().to_string())

print("\n=== query_id별 포함/제외 ===")
summary = df.groupby("query_id")["gold_status"].value_counts().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())

# ── query_id별 grade 분포 상세 테이블 ──────────────────────────────────
# "excluded" = 3개 judge 합의 실패로 goldset에서 제외된 문서
#              (LLM API 실패와 무관 — judge 파일의 Failed 건수와 다름)
print("\n[query_id별 relevance_grade 분포]")

df["_label"] = df.apply(
    lambda r: f"grade {int(r['final_grade'])}" if r["gold_status"] == "included" else "excluded",
    axis=1,
)
pivot = df.groupby(["query_id", "_label"]).size().unstack(fill_value=0)
df.drop(columns=["_label"], inplace=True)

grade_cols = [f"grade {i}" for i in range(4) if f"grade {i}" in pivot.columns]
col_order  = grade_cols + (["excluded"] if "excluded" in pivot.columns else [])
pivot      = pivot.reindex(columns=col_order, fill_value=0)
pivot["total"] = pivot.sum(axis=1)

col_w = 9
qid_w = 10
header = f"  {'query_id':>{qid_w}}" + "".join(f"  {c:>{col_w}}" for c in col_order) + f"  {'total':>{col_w}}"
sep    = "-" * (len(header))
print(header)
print(sep)
for qid, row in pivot.iterrows():
    vals = "".join(f"  {int(row[c]):>{col_w}}" for c in col_order)
    print(f"  {qid:>{qid_w}}{vals}  {int(row['total']):>{col_w}}")

=== confidence 분포 ===
confidence
HIGH      2030
MEDIUM    1292
LOW         38

=== final_grade 분포 (포함된 문서) ===
final_grade
0.0     191
1.0    1865
2.0     777
3.0     489

=== query_id별 포함/제외 ===
gold_status  excluded  included  total
query_id                              
S01                 1        79     80
S02                 0        80     80
S03                 0        80     80
S04                 0        80     80
S05                 4        76     80
S06                 1        79     80
S07                 0        80     80
S08                 0        80     80
S09                 1        79     80
S10                 3        77     80
S11                 1        79     80
S12                 2        78     80
S13                 5        75     80
S14                 0        80     80
S15                 0        80     80
S16                 2        78     80
S17                 2        78     80
S18                 3        77     80
S19                 2   

## 6. query_id별 grade 2~3 문서 충분성 점검

**최소 기준**: query_id별 grade ≥ 2 문서 3개 이상

In [9]:
MIN_GRADE  = 2
MIN_COUNT  = 3

high_grade = included[included["final_grade"] >= MIN_GRADE]
grade_count = high_grade.groupby("query_id").size().rename(f"grade>={MIN_GRADE}_count")

# 전체 query_id 기준으로 merge (없으면 0)
all_qids = pd.Series(sorted(df["query_id"].unique()), name="query_id")
check = all_qids.to_frame().merge(grade_count.reset_index(), on="query_id", how="left").fillna(0)
check[f"grade>={MIN_GRADE}_count"] = check[f"grade>={MIN_GRADE}_count"].astype(int)
check["sufficient"] = check[f"grade>={MIN_GRADE}_count"] >= MIN_COUNT

print(f"=== grade ≥ {MIN_GRADE} 문서 수 (최소 {MIN_COUNT}개 기준) ===")
print(check.to_string(index=False))

insufficient = check[~check["sufficient"]]
if insufficient.empty:
    print(f"\n✅ 모든 query_id가 최소 기준({MIN_COUNT}개) 충족")
else:
    print(f"\n⚠️  기준 미달 query_id ({len(insufficient)}개) → 재검토 필요:")
    print(insufficient.to_string(index=False))

=== grade ≥ 2 문서 수 (최소 3개 기준) ===
query_id  grade>=2_count  sufficient
     S01               9        True
     S02               0       False
     S03              33        True
     S04              29        True
     S05              27        True
     S06              56        True
     S07              34        True
     S08              27        True
     S09              29        True
     S10              56        True
     S11              14        True
     S12              54        True
     S13              22        True
     S14              12        True
     S15              11        True
     S16              28        True
     S17              40        True
     S18              65        True
     S19              64        True
     S20              37        True
     S21              45        True
     S22              28        True
     S23              20        True
     S24              26        True
     S25               8        True
    

## 7. 제외된 문서 상세 (LOW 판정)

In [10]:
excluded = df[df["gold_status"] == "excluded"].copy()
excluded["grades"] = excluded.apply(lambda r: [r.grade_j1, r.grade_j2, r.grade_j3], axis=1)

print(f"제외 문서 총 {len(excluded)}건")
print("\n=== query_id별 제외 수 ===")
print(excluded["query_id"].value_counts().sort_index().to_string())

print("\n=== 제외 문서 샘플 (상위 10개) ===")
pd.set_option("display.max_colwidth", 25)
excluded[["query_id", "isbn", "title", "grades"]]

제외 문서 총 38건

=== query_id별 제외 수 ===
query_id
S01    1
S05    4
S06    1
S09    1
S10    3
S11    1
S12    2
S13    5
S16    2
S17    2
S18    3
S19    2
S20    1
S25    1
S26    1
S28    1
S31    1
S32    2
S34    1
S37    1
S40    2

=== 제외 문서 샘플 (상위 10개) ===


,query_id,isbn,title,grades
106,S13,9788928072774,세상은 넓고 맞을 놈은 많다 5,"[2, 0, 1]"
272,S11,9788937477317,꽤 낙천적인 아이 - 의 젊은 작가 50,"[1, 2, 3]"
329,S13,9788946413627,TV 동화 행복한 세상 2,"[2, 3, 1]"
379,S20,9788952213181,"하리하라, 미드에서 과학을 보다 - 하...","[2, 1, 3]"
402,S17,9788954440011,소녀를 위한 페미니즘 - 과모음 청소년...,"[2, 1, 3]"
463,S05,9788956582498,내 삶의 메아리 - 채홍석 시문집,"[3, 1, 2]"
585,S18,9788959138258,한뼘한뼘 - 마음을 다독이는 힐링토끼의...,"[2, 3, 1]"
656,S40,9788960908833,웨하스 소년 - 산책 짧은 소설,"[1, 3, 2]"
718,S01,9788963652337,쇼펜하우어 인생론,"[2, 1, 3]"
764,S05,9788965770459,내 영혼의 메아리 - 춘추 기획수필선 3,"[2, 3, 1]"


## 8. 최종 Goldset 저장

In [11]:
# 포함된 문서에 원본 메타데이터 결합
included_keys = set(zip(included["isbn"], included["query_id"]))

final_records = []
for key in sorted(included_keys):
    isbn, query_id = key
    base  = meta[key].copy()
    row   = df[(df["isbn"] == isbn) & (df["query_id"] == query_id)].iloc[0]

    base["final_grade"] = int(row["final_grade"])
    base["confidence"]  = row["confidence"]
    base["grade_j1"]    = int(row["grade_j1"])
    base["grade_j2"]    = int(row["grade_j2"])
    base["grade_j3"]    = int(row["grade_j3"])
    final_records.append(base)

out_path = DATA_DIR / "goldset_final.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_records, f, ensure_ascii=False, indent=2)

# out_jsonl = DATA_DIR / "goldset_final.jsonl"
# with open(out_jsonl, "w", encoding="utf-8") as f:
#     for r in final_records:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"저장 완료: {out_path}  ({len(final_records):,}건)")
# print(f"저장 완료: {out_jsonl}")

저장 완료: ../dataset/goldset_final.json  (3,322건)


## 9. 최종 Goldset 검증

In [12]:
final_df = pd.DataFrame(final_records)

print("=== 최종 Goldset 요약 ===")
print(f"총 문서 수    : {len(final_df):,}")
print(f"query_id 수   : {final_df['query_id'].nunique()}")

print("\n=== query_id별 grade 분포 ===")
pivot = final_df.groupby(["query_id", "final_grade"]).size().unstack(fill_value=0)
pivot.columns = [f"grade_{c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
if "grade_2" in pivot.columns and "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"] + pivot["grade_3"]
elif "grade_2" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_2"]
elif "grade_3" in pivot.columns:
    pivot["grade_2+3"] = pivot["grade_3"]
print(pivot.to_string())

print("\n=== confidence 분포 ===")
print(final_df["confidence"].value_counts().to_string())

=== 최종 Goldset 요약 ===
총 문서 수    : 3,322
query_id 수   : 42

=== query_id별 grade 분포 ===
          grade_0  grade_1  grade_2  grade_3  total  grade_2+3
query_id                                                      
S01             1       69        8        1     79          9
S02            14       66        0        0     80          0
S03             4       43       22       11     80         33
S04             1       50       19       10     80         29
S05             4       45       16       11     76         27
S06             0       23       45       11     79         56
S07             7       39       27        7     80         34
S08             3       50       22        5     80         27
S09             2       48       21        8     79         29
S10             0       21       31       25     77         56
S11             4       61       10        4     79         14
S12             1       23        7       47     78         54
S13             0       53      

In [13]:
REVIEW_FIELDS = [
    "query_id", "isbn", "title", "author", "publish_date", "page",
    "large_cate", "mid_cate", "small_cate",
    "book_intro", "book_index", "review_text", 
    "confidence", "grade_j1", "grade_j2", "grade_j3", "final_grade", "llm_raw_output"
]


def get_min_rank(item):
    ri = item.get("retrieval_info", [])
    return min((r.get("rank", 9999) for r in ri), default=None)


def build_review_entry(item):
    min_rank = get_min_rank(item)
    entry = {k: item.get(k) for k in REVIEW_FIELDS}
    entry["semantic_query"]     = (item.get("rag_query") or {}).get("semantic_query", "")
    entry["min_retrieval_rank"] = min_rank
    entry["corrected_grade"]    = None   # 검수자가 grade 수정 시 기입 (0~3)
    return entry


# 1순위만 저장: MEDIUM × grade 2~3
review_items = [
    build_review_entry(item)
    for item in final_records
    if item.get("confidence") == "MEDIUM" and item.get("final_grade", 0) >= 2
]

# 정렬: query_id → min_retrieval_rank
review_items.sort(key=lambda x: (
    x.get("query_id", ""),
    x.get("min_retrieval_rank") or 9999,
))

# 저장
out_path = DATA_DIR / "goldset_review.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(review_items, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path}  ({len(review_items):,}건)")
print(f"  MEDIUM × grade 2~3  (1순위 검수 대상)")
print(f"\n  corrected_grade 필드: null → 검수 후 수정할 grade(0~3) 기입")

저장 완료: ../dataset/goldset_review.json  (631건)
  MEDIUM × grade 2~3  (1순위 검수 대상)

  corrected_grade 필드: null → 검수 후 수정할 grade(0~3) 기입


In [14]:
import math

mid = math.ceil(len(review_items) / 2)   # 320 → 160 / 160
half1 = review_items[:mid]
half2 = review_items[mid:]

def to_csv_df(items):
    return pd.DataFrame([
        {"isbn": r["isbn"], "grade": r["final_grade"], "corrected_grade": r["corrected_grade"]}
        for r in items
    ])

csv1 = DATA_DIR / "goldset_review_1.csv"
csv2 = DATA_DIR / "goldset_review_2.csv"
to_csv_df(half1).to_csv(csv1, index=False, encoding="utf-8-sig")
to_csv_df(half2).to_csv(csv2, index=False, encoding="utf-8-sig")

print(f"저장 완료")
print(f"  {csv1.name}  ({len(half1)}건)  query_id {half1[0]['query_id']} ~ {half1[-1]['query_id']}")
print(f"  {csv2.name}  ({len(half2)}건)  query_id {half2[0]['query_id']} ~ {half2[-1]['query_id']}")

저장 완료
  goldset_review_1.csv  (316건)  query_id S01 ~ S20
  goldset_review_2.csv  (315건)  query_id S21 ~ S42


## 11. 검수 파일 생성

`goldset_review.json` — 우선순위별 검수 대상을 하나의 파일로 통합

- `review_priority`: 1 / 2 / 3
- `min_retrieval_rank`: 검색 소스 중 가장 높은 rank (낮을수록 상위)
- `rank_thresholds`: 해당 항목이 포함되는 threshold 목록 (예: `[5, 10, 20]`)
- `semantic_query`: rag_query에서 추출한 검색 의도
- 2순위는 `min_rank ≤ 20` 이하 전체 포함 (threshold는 `rank_thresholds`로 필터링)

In [15]:
def get_min_rank(item):
    ri = item.get("retrieval_info", [])
    return min((r.get("rank", 9999) for r in ri), default=None)


# 1순위: MEDIUM × grade 2~3
p1 = [d for d in final_records if d.get("confidence") == "MEDIUM" and d.get("final_grade", 0) >= 2]

# 2순위: grade 1 전체 (threshold별 필터링)
grade1_all = [d for d in final_records if d.get("final_grade") == 1]

# 3순위: LOW (excluded df는 위 셀에서 정의됨)
p3_cnt = len(excluded)

print("=== 검수 우선순위 갯수 ===")
print(f"1순위  MEDIUM × grade 2~3         : {len(p1):>5}건")
print()
for t in [5, 10, 20]:
    n = sum(1 for d in grade1_all if get_min_rank(d) is not None and get_min_rank(d) <= t)
    print(f"2순위  grade 1 × min_rank ≤ {t:2d}   : {n:>5}건")
print()
print(f"3순위  LOW (합의 실패, excluded)   : {p3_cnt:>5}건")

=== 검수 우선순위 갯수 ===
1순위  MEDIUM × grade 2~3         :   631건

2순위  grade 1 × min_rank ≤  5   :    67건
2순위  grade 1 × min_rank ≤ 10   :   163건
2순위  grade 1 × min_rank ≤ 20   :   365건

3순위  LOW (합의 실패, excluded)   :    38건


## 10. 검수 우선순위 갯수 확인

| 우선순위 | 조건 | 설명 |
|----------|------|------|
| 1순위 | MEDIUM × grade 2~3 | judge 2/3 합의, 사람 검수 필요 |
| 2순위 | grade 1 × retrieval 상위권 | Hard Negative 후보 (threshold 결정 필요) |
| 3순위 | LOW (합의 실패) | judge 3명 모두 다른 grade |

## 12. 기본 검수 — (1,2,2) 패턴 항목

규칙: judge 3명 성적이 정렬 시 `[1, 2, 2]`인 항목만 대상  
- **셀 B** 실행 → `inspection_basic.json` 저장 → 검수자가 `corrected_grade` 입력  
- **셀 C** 실행 → 결과 반영 (`adjusted_records` 생성)  

| corrected_grade | 처리 |
|---|---|
| `None` (미수정) | 변경 없음 (grade 2 유지) |
| `0` 또는 `1` | final_grade = 1 로 내림 |

In [16]:
# ── 기본 검수 대상 추출 및 검수 파일 생성 ──────────────────────────────
BASIC_INSPECTION_FIELDS = [
    "query_id", "isbn", "title", "author", "publish_date", "page",
    "large_cate", "mid_cate", "small_cate", 
    "book_intro", "book_index", "review_text",
    "grade_j1", "grade_j2", "grade_j3", "final_grade", "confidence",
    "llm_raw_output"
]

def _get_min_rank(item):
    return min((r.get("rank", 9999) for r in item.get("retrieval_info", [])), default=None)

basic_inspection_items = []
for item in final_records:
    grades = sorted([item.get("grade_j1", 0), item.get("grade_j2", 0), item.get("grade_j3", 0)])
    if grades != [1, 3, 3] and grades != [1, 2, 2]:
        continue
    entry = {k: item.get(k) for k in BASIC_INSPECTION_FIELDS}
    entry["semantic_query"]     = (item.get("rag_query") or {}).get("semantic_query", "")
    entry["min_retrieval_rank"] = _get_min_rank(item)
    entry["corrected_grade"]    = None  # 검수자 입력: relevant → 생략, not relevant → 1
    basic_inspection_items.append(entry)

basic_inspection_items.sort(key=lambda x: (x["query_id"], x.get("min_retrieval_rank") or 9999))

out_path = DATA_DIR / "inspection_basic.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(basic_inspection_items, f, ensure_ascii=False, indent=2)

print(f"(1,3,3) 패턴 항목: {len(basic_inspection_items):,}건")
print(f"저장 완료: {out_path}")
print()

# query_id별 분포
from collections import Counter as _Counter
qid_cnt = _Counter(x["query_id"] for x in basic_inspection_items)
for qid in sorted(qid_cnt):
    print(f"  {qid}: {qid_cnt[qid]}건")

(1,3,3) 패턴 항목: 288건
저장 완료: ../dataset/inspection_basic.json

  S01: 2건
  S03: 7건
  S04: 10건
  S05: 6건
  S06: 12건
  S07: 6건
  S08: 11건
  S09: 12건
  S10: 10건
  S11: 6건
  S12: 2건
  S13: 9건
  S14: 6건
  S15: 6건
  S16: 8건
  S17: 8건
  S18: 6건
  S19: 6건
  S20: 10건
  S21: 19건
  S22: 10건
  S23: 9건
  S24: 6건
  S25: 4건
  S26: 8건
  S27: 8건
  S28: 4건
  S29: 6건
  S30: 2건
  S31: 7건
  S32: 5건
  S33: 7건
  S34: 7건
  S35: 8건
  S36: 5건
  S37: 2건
  S38: 3건
  S39: 10건
  S40: 5건
  S41: 3건
  S42: 7건


In [21]:
# ── 기본 검수 결과 적용 (검수 완료 후 실행) ───────────────────────────
with open(DATA_DIR / "goldset_final_corrected.json", encoding="utf-8") as f:
    _basic_review = json.load(f)

basic_corrections = {
    (r["isbn"], r["query_id"]): r["corrected_grade"]
    for r in _basic_review
    if r.get("corrected_grade") is not None
}

adjusted_records = []
n_lowered = 0
for item in final_records:
    key = (item["isbn"], item["query_id"])
    corrected = basic_corrections.get(key)
    new_item = dict(item)
    if corrected is not None and corrected < 2:
        new_item["final_grade"] = 1
        new_item["corrected_by"] = "basic_inspection"
        n_lowered += 1
    adjusted_records.append(new_item)

print(f"기본 검수 적용 완료")
print(f"  grade 1로 내린 항목: {n_lowered}건")
print(f"  변경 없음:           {len(adjusted_records) - n_lowered}건")

기본 검수 적용 완료
  grade 1로 내린 항목: 0건
  변경 없음:           3322건


## 13. 예외 검수 — 특정 시나리오 grade 1 상위 후보

대상 시나리오: **S01, S02, S25, S30, S37**  
규칙: 위 시나리오에서 `final_grade == 1`인 항목을 검색 순위 순으로 확인  
- **셀 E** 실행 → `inspection_exception.json` 저장 → 검수자가 `corrected_grade` 입력  
- **셀 F** 실행 → 결과 반영 (`final_adjusted` 생성)  

| corrected_grade | 처리 |
|---|---|
| `None` (미수정) | 변경 없음 (grade 1 유지) |
| `2` 이상 | final_grade = 2 로 올림 |

In [22]:
# ── 예외 검수 대상 추출 및 검수 파일 생성 ──────────────────────────────
# ※ 셀 C (기본 검수 결과 적용) 이후에 실행할 것
# 조건: 예외 시나리오 & final_grade == 1 & grade 패턴이 [1, 1, 2] 인 항목
EXCEPTION_SCENARIOS = {"S01", "S02", "S25", "S30", "S37"}

_base = adjusted_records if "adjusted_records" in dir() else final_records

exception_inspection_items = []
for item in _base:
    if item.get("query_id") not in EXCEPTION_SCENARIOS:
        continue
    if item.get("final_grade") != 1:
        continue
    grades = sorted([item.get("grade_j1", 0), item.get("grade_j2", 0), item.get("grade_j3", 0)])
    if grades != [1, 1, 2]:          # (1,1,2) 패턴만 대상
        continue
    entry = {k: item.get(k) for k in BASIC_INSPECTION_FIELDS}
    entry["semantic_query"]     = (item.get("rag_query") or {}).get("semantic_query", "")
    entry["min_retrieval_rank"] = _get_min_rank(item)
    entry["corrected_grade"]    = None  # 검수자 입력: relevant → 2, 아니면 생략
    exception_inspection_items.append(entry)

exception_inspection_items.sort(
    key=lambda x: (x["query_id"], x.get("min_retrieval_rank") or 9999)
)

out_path = DATA_DIR / "inspection_exception.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(exception_inspection_items, f, ensure_ascii=False, indent=2)

print(f"예외 검수 대상 (1,1,2 패턴): {len(exception_inspection_items):,}건")
print(f"{'시나리오':>10}  {'(1,1,2) 검수대상':>16}  {'현재 grade2~3':>13}")
print("-" * 48)
for qid in sorted(EXCEPTION_SCENARIOS):
    cnt  = sum(1 for x in exception_inspection_items if x["query_id"] == qid)
    g23  = sum(1 for x in _base if x["query_id"] == qid and x.get("final_grade", 0) >= 2)
    print(f"  {qid:>8}  {cnt:>16}  {g23:>13}")
print(f"\n저장 완료: {out_path}")

예외 검수 대상 (1,1,2 패턴): 20건
      시나리오      (1,1,2) 검수대상    현재 grade2~3
------------------------------------------------
       S01                 6              9
       S02                 0              0
       S25                 3              8
       S30                 0              4
       S37                11              4

저장 완료: ../dataset/inspection_exception.json


In [23]:
# ── 예외 검수 결과 적용 (검수 완료 후 실행) ───────────────────────────
with open(DATA_DIR / "inspection_exception.json", encoding="utf-8") as f:
    _exception_review = json.load(f)

exception_corrections = {
    (r["isbn"], r["query_id"]): r["corrected_grade"]
    for r in _exception_review
    if r.get("corrected_grade") is not None and r["corrected_grade"] >= 2
}

_base2 = adjusted_records if "adjusted_records" in dir() else final_records

final_adjusted = []
n_raised = 0
for item in _base2:
    key = (item["isbn"], item["query_id"])
    corrected = exception_corrections.get(key)
    new_item = dict(item)
    if corrected is not None:
        new_item["final_grade"] = 2
        new_item["corrected_by"] = "exception_inspection"
        n_raised += 1
    final_adjusted.append(new_item)

print(f"예외 검수 적용 완료")
print(f"  grade 2로 올린 항목: {n_raised}건")

예외 검수 적용 완료
  grade 2로 올린 항목: 0건


In [24]:
# ── 최종 제외 — 시나리오별 Main / Conditional / Excluded 분류 ──────────
# ※ 셀 F (예외 검수 결과 적용) 이후에 실행할 것
from collections import defaultdict as _defaultdict

_records = final_adjusted if "final_adjusted" in dir() else final_records

grade23_per_qid = _defaultdict(int)
for item in _records:
    if item.get("final_grade", 0) >= 2:
        grade23_per_qid[item["query_id"]] += 1

all_qids = sorted(set(item["query_id"] for item in _records))

main_eval        = [(q, grade23_per_qid[q]) for q in all_qids if grade23_per_qid[q] >= 10]
conditional_eval = [(q, grade23_per_qid[q]) for q in all_qids if 5 <= grade23_per_qid[q] <= 9]
excluded_eval    = [(q, grade23_per_qid[q]) for q in all_qids if grade23_per_qid[q] < 5]

print("=== 최종 제외 판단 결과 ===\n")
print(f"[Main Eval]         {len(main_eval)}개 시나리오  (grade 2~3 ≥ 10개)")
for q, cnt in main_eval:
    print(f"  {q}: {cnt}건")

print(f"\n[Conditional Eval]  {len(conditional_eval)}개 시나리오  (grade 2~3 5~9개)")
for q, cnt in conditional_eval:
    print(f"  {q}: {cnt}건")

print(f"\n[Excluded]          {len(excluded_eval)}개 시나리오  (grade 2~3 < 5개)")
for q, cnt in excluded_eval:
    print(f"  {q}: {cnt}건")

# eval_status 컬럼 추가 후 저장
status_map = (
    {q: "main_eval"        for q, _ in main_eval}
  | {q: "conditional_eval" for q, _ in conditional_eval}
  | {q: "excluded"         for q, _ in excluded_eval}
)
for item in _records:
    item["eval_status"] = status_map.get(item["query_id"], "excluded")

out_path = DATA_DIR / "goldset_final_v2.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(_records, f, ensure_ascii=False, indent=2)

total_main = sum(1 for x in _records if x["eval_status"] == "main_eval")
total_cond = sum(1 for x in _records if x["eval_status"] == "conditional_eval")
total_excl = sum(1 for x in _records if x["eval_status"] == "excluded")
print(f"\n저장 완료: {out_path}  ({len(_records):,}건)")
print(f"  main_eval:        {total_main:,}건  ({len(main_eval)}개 시나리오)")
print(f"  conditional_eval: {total_cond:,}건  ({len(conditional_eval)}개 시나리오)")
print(f"  excluded:         {total_excl:,}건  ({len(excluded_eval)}개 시나리오)")

=== 최종 제외 판단 결과 ===

[Main Eval]         37개 시나리오  (grade 2~3 ≥ 10개)
  S03: 33건
  S04: 29건
  S05: 27건
  S06: 56건
  S07: 34건
  S08: 27건
  S09: 29건
  S10: 56건
  S11: 14건
  S12: 54건
  S13: 22건
  S14: 12건
  S15: 11건
  S16: 28건
  S17: 40건
  S18: 65건
  S19: 64건
  S20: 37건
  S21: 45건
  S22: 28건
  S23: 20건
  S24: 26건
  S26: 48건
  S27: 16건
  S28: 34건
  S29: 53건
  S31: 28건
  S32: 40건
  S33: 20건
  S34: 28건
  S35: 57건
  S36: 40건
  S38: 10건
  S39: 18건
  S40: 35건
  S41: 11건
  S42: 46건

[Conditional Eval]  2개 시나리오  (grade 2~3 5~9개)
  S01: 9건
  S25: 8건

[Excluded]          3개 시나리오  (grade 2~3 < 5개)
  S02: 0건
  S30: 4건
  S37: 4건



저장 완료: ../dataset/goldset_final_v2.json  (3,322건)
  main_eval:        2,925건  (37개 시나리오)
  conditional_eval: 158건  (2개 시나리오)
  excluded:         239건  (3개 시나리오)


In [ ]:
# Excluded 시나리오: S02, S30, S37 제외 후 저장
excluded_scenarios = ["S02", "S30", "S37"]
excluded_records = [item for item in adjusted_records if item.get("query_id") not in excluded_scenarios]
print(f"제외된 시나리오: {excluded_scenarios}")
print(f"제외 후 남은 항목: {len(excluded_records)}건")

# 최종 결과 저장
with open(DATA_DIR / "goldset_final_v2.json", "w", encoding="utf-8") as f:
    json.dump(excluded_records, f, ensure_ascii=False, indent=2)

제외된 시나리오: ['S02', 'S30', 'S37']
제외된 항목 수: 3083건
